In [1]:
import os

from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.world_entity import Body
print(os.environ.get("ROS_VERSION"))
from pycram.testing import setup_world
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.robots.hsrb import HSRB
from pycram.datastructures.dataclasses import Context

world = setup_world()
pr2_view = PR2.from_world(world)
# pr2_view = HSRB.from_world(world)
context = Context(world, pr2_view)

2


Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


In [2]:
# context.world.state
print(world.get_body_by_name('milk.stl'))
print(world.get_body_by_name('milk.stl')._semantic_annotations)
# body_prefixed_names = []
# body_names = []
# for body in world.kinematic_structure.nodes():
#     body_prefixed_names.append(body.name)
#     body_names.append(body.name.name)

Body(name=PrefixedName(name='milk.stl', prefix=None), index=113, collision_config=CollisionCheckingConfig(buffer_zone_distance=None, violated_distance=0.0, disabled=None, max_avoided_bodies=1), temp_collision_config=None)
{Milk(class_label=None, name=PrefixedName(name='Milk_1', prefix=None), body=Body(name=PrefixedName(name='milk.stl', prefix=None), index=113, collision_config=CollisionCheckingConfig(buffer_zone_distance=None, violated_distance=0.0, disabled=None, max_avoided_bodies=1), temp_collision_config=None))}


In [3]:
# milk_json  = world.get_body_by_name('milk.stl').to_json()

## visualize the 'world' in rviz2 || open rviz2 first

In [4]:
from semantic_digital_twin.adapters.viz_marker import VizMarkerPublisher
import threading
import rclpy
rclpy.init()

node = rclpy.create_node("semantic_digital_twin")
thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
thread.start()

In [5]:
viz = VizMarkerPublisher(world=world, node=node)

## Perform/Run  Actions/Plans/Designators

In [6]:
from flask import Flask, jsonify, request
import threading
import traceback
from pycram.robot_plans import *
from pycram.datastructures.pose import PoseStamped
from pycram.datastructures.enums import Arms
from pycram.language import SequentialPlan

app = Flask(__name__)

@app.route("/pre_names")
def pre_names():
    body_prefixed_names = []
    for body in world.kinematic_structure.nodes():
        body_prefixed_names.append(body.name)
    data = body_prefixed_names
    return jsonify(data)

@app.route("/names")
def names():
    body_names = []
    for body in world.kinematic_structure.nodes():
        body_names.append(body.name)
    data = body_names
    return jsonify(data)

@app.route("/semantic_annotations")
def semantic_annotations():
    return jsonify(world.semantic_annotations)

@app.route("/runner", methods=["POST"])
def submit():
    try:
        if not request.is_json:
            return jsonify({"error": "Request must be JSON"}), 400

        data = request.get_json()
        action = data.get("action")
        bod = data.get("entity")
        if ".stl" in bod:
            bod = bod
        else:
            bod = bod+".stl"

        if not action or not bod:
            return jsonify({
                "error": "Missing required fields",
                "required": ["action", "entity"],
                "received": data
            }), 400

        cls = globals()[action] # need to chnage for multiple action names
        body_world = world.get_body_by_name(bod)   # likely crash point if bod is wrong/None
        if body_world is None:
            return jsonify({"error": f"Unknown entity '{bod}'"}), 400

        if "Transport" in action:
            obj = cls(
                body_world,
                PoseStamped.from_list([4.9, 3.3, 0.8], frame=world.root),
                Arms.LEFT
            )
        elif "PickUp" in action:
            obj = cls(
                body_world,
                Arms.LEFT,
                GraspDescription(approach_direction=ApproachDirection.FRONT)
            )
        else:
            obj = cls(
                body_world,
                target_location=PoseStamped.from_list([5.1, 3.3, 0.75], [0, 0, 1, 1]), arm=Arms.LEFT
            )

        # trans = TransportActionDescription(
        #     body_world,
        #     PoseStamped.from_list([4.9, 3.3, 0.8], frame=world.root),
        #     Arms.LEFT
        # )

        plan = SequentialPlan(context, obj)

        from pycram.process_module import simulated_robot
        with simulated_robot:
            plan.perform()

        return jsonify({"message": "OK", "received": data}), 200

    except Exception as e:
        tb = traceback.format_exc()
        print(tb)  # shows up in your console
        return jsonify({"error": str(e), "traceback": tb}), 500

@app.route("/shutdown", methods=["POST"])
def shutdown():
    func = request.environ.get("werkzeug.server.shutdown")
    if func is None:
        raise RuntimeError("Not running with the Werkzeug Server")
    func()
    return "Server shutting down..."

def run_flask():
    app.run(host="127.0.0.1", port=5001, debug=False, use_reloader=False)

threading.Thread(target=run_flask).start()


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:_internal.py::97 _log WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5001
INFO:_internal.py::97 _log Press CTRL+C to quit


In [ ]:
from pycram.robot_plans import *
from pycram.datastructures.pose import PoseStamped
from pycram.datastructures.enums import Arms
from pycram.language import SequentialPlan

navigate = NavigateActionDescription(target_location=PoseStamped.from_list([1, 2, 0]), keep_joint_states= [True, False])
park = ParkArmsActionDescription([Arms.BOTH])
pickup = PickUpActionDescription(world.get_body_by_name('milk.stl'),Arms.LEFT, GraspDescription(approach_direction=ApproachDirection.FRONT))
place = PlaceActionDescription(world.get_body_by_name('milk.stl'),target_location=PoseStamped.from_list([5.1, 3.3, 0.75], [0, 0, 1, 1]), arm=Arms.LEFT)
trans = TransportActionDescription(world.get_body_by_name("milk.stl"),
                                                 PoseStamped.from_list([4.9, 3.3, 0.8], frame=world.root), Arms.LEFT)


In [ ]:
plan = SequentialPlan(context, navigate, trans)

In [ ]:
next(iter(navigate))

In [ ]:
navigate.performable

In [ ]:
plan.plan_graph.nodes()

In [ ]:
plan.current_plan

In [ ]:
plan.root.children

In [ ]:
from pycram.process_module import simulated_robot

with simulated_robot:
    plan.perform()

In [ ]:
context.world.state

In [ ]:
# context

In [ ]:
nodes = plan.plan_graph.nodes()
nodes

In [ ]:
for n in nodes:
    print(type(n))

In [ ]:
nodes[3]

In [ ]:
from pycram.datastructures.enums import TorsoState, Arms


## pycram_bullet_world_demo

In [ ]:
import os

from semantic_digital_twin.adapters.mesh import STLParser
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import Container
from semantic_digital_twin.adapters.procthor.procthor_semantic_annotations import Milk, Bowl, Spoon
from semantic_digital_twin.spatial_types import TransformationMatrix
from semantic_digital_twin.world_description.connections import FixedConnection

from pycram.datastructures.dataclasses import Context
from pycram.datastructures.enums import TorsoState, Arms
from pycram.datastructures.pose import PoseStamped
from pycram.language import SequentialPlan
from pycram.process_module import simulated_robot
from pycram.robot_plans import MoveTorsoActionDescription, TransportActionDescription, PickUpActionDescription
from pycram.robot_plans import ParkArmsActionDescription
from pycram.testing import setup_world

In [ ]:
world = setup_world()

In [ ]:
spoon = STLParser(os.path.join(os.getcwd(), "..", "resources", "objects", "spoon.stl")).parse()
bowl = STLParser(os.path.join(os.getcwd(), "..", "resources", "objects", "bowl.stl")).parse()

In [ ]:
with world.modify_world():
    world.merge_world_at_pose(bowl, TransformationMatrix.from_xyz_quaternion(2.4, 2.2, 1, reference_frame=world.root))
    connection = FixedConnection(parent=world.get_body_by_name("cabinet10_drawer_top"), child=spoon.root)
    world.merge_world(spoon, connection)

In [ ]:
try:
    import rclpy

    rclpy.init()
    from semantic_digital_twin.adapters.viz_marker import VizMarkerPublisher

    v = VizMarkerPublisher(world, rclpy.create_node("viz_marker"))
except ImportError:
    pass

In [ ]:
pr2 = PR2.from_world(world)
context = Context.from_world(world)

In [ ]:
from pycram.robot_description import RobotDescription


In [ ]:
context.robot

In [ ]:
context.world.state

In [ ]:
with world.modify_world():
    world_reasoner = WorldReasoner(world)
    world_reasoner.reason()
    world.add_semantic_annotations([Bowl(body=world.get_body_by_name("bowl.stl")),
                                    Spoon(body=world.get_body_by_name("spoon.stl"))
                                    ])

In [ ]:
world_reasoner.reason()

In [ ]:
plan = SequentialPlan(context,
                      ParkArmsActionDescription(Arms.BOTH),
                      MoveTorsoActionDescription(TorsoState.HIGH),
                      TransportActionDescription(world.get_body_by_name("milk.stl"),
                                                 PoseStamped.from_list([4.9, 3.3, 0.8], frame=world.root), Arms.LEFT),
                      TransportActionDescription(world.get_body_by_name("spoon.stl"),
                                                 PoseStamped.from_list([5.1, 3.3, 0.75], [0, 0, 1, 1], frame=world.root), Arms.LEFT),
                      TransportActionDescription(world.get_body_by_name("bowl.stl"),
                                                 PoseStamped.from_list([5, 3.3, 0.75], frame=world.root), Arms.LEFT))


In [ ]:
with simulated_robot:
    plan.perform()

In [ ]:
context.world.state

In [ ]:
# node.destroy_node()
# rclpy.shutdown()

In [ ]:
# world_reasoner

In [ ]:
world_reasoner.reason()

In [ ]:
from inspect import signature
from dataclasses import dataclass

@dataclass(frozen=True)
class Sign:
    test1 : int
    test2 : str

In [ ]:
signature(Sign)

In [ ]:
signature(Sign).parameters

In [ ]:
signature(Sign.__init__).parameters

In [ ]:
signature(world.__init__).parameters

In [ ]:
import inspect

def get_public_methods_with_docstrings(cls):
    methods = []
    for name, member in inspect.getmembers(cls, predicate=inspect.isfunction):
        if not name.startswith("get"):
            methods.append({
                "method": name,
                "docstring": inspect.getdoc(member)
            })
    return methods


In [ ]:
print(get_public_methods_with_docstrings(world))

In [ ]:
inspect.getmembers(world.__class__, predicate=inspect.isfunction)

In [ ]:
def find_kinematic_structure_entities(world : World):
    return world.kinematic_structure_entities
def find_body_by_name(world : World, name : str):
    return world.get_body_by_name(name)
def find_semantic_annotations_of_body(body: Body):
    return body._semantic_annotations

In [ ]:
body = find_body_by_name(world, "milk.stl")
find_semantic_annotations_of_body(body)